### Kansas City, St. Louis, and Springfield

In [ ]:
import pandas as pd
import numpy as np
import re
from difflib import get_close_matches

In [207]:
kc = pd.read_csv(
    'auction_data/kcmo.csv',
    dtype={"Zip":"string"})
stl = pd.read_csv('auction_data/stl.csv')
spfd = pd.read_csv('auction_data/springfield.csv')

#### Add unique ID's & Market info to each county csv

In [208]:
#KC

# Create APN column
kc["APN"] = (
    "KCMO."
    + kc["Address"]
        .fillna("")
        .str.replace(" ", "_",regex = False)
        .str.upper()
    +"_"
    + kc["Zip"].astype(str).str.replace(r"\.0$", "", regex=True)
)

#Create Market column
kc["Market"] = "Kansas City"


In [209]:
kc.head()

,County,Date,Time,Address,City,Zip,Owner Name,Mailing Address,Bedrooms,Bathrooms,...,Opening Bid,Assessed Value,Proj. Opening Bid,Original Loan Amount,APN Number,Status,Action,Comments,APN,Market
0,Jackson County,07-13-2026,10:00,4070 SW Lido Dr,Lees Summit,64082,Jason T Medley and Heather Medley,"4070 SW Lido Dr, Lees Summit, MO 64082",3.0,2.0,...,NaN,415800.0,327328.0,330687.0,70-930-01-46-00-0-00-000,Canceled,NaN,NaN,KCMO.4070_SW_LIDO_DR_64082,Kansas City
1,Cass County,07-13-2026,10:00,306 W Walnut St,Archie,64725,Cody S Thompson,"306 W Walnut St, Archie, MO 64725",2.0,NaN,...,69600.00,165700.0,153317.0,157070.0,77500,Active,NaN,NaN,KCMO.306_W_WALNUT_ST_64725,Kansas City
2,Clay County,07-13-2026,10:40,800 Southbrook Pkwy,Kearney,64060,Kurtis L. Springer,"800 Southbrook Pkwy, Kearney, MO 64060",3.0,2.0,...,NaN,315000.0,112584.0,137710.0,07-918-00-15-005.00,Canceled,NaN,NaN,KCMO.800_SOUTHBROOK_PKWY_64060,Kansas City
3,Clay County,07-13-2026,10:40,10804 N Randolph Ave,Kansas City,64157,Ronald G Smith and Janet M Smith,"10804 N Randolph Ave, Kansas City, MO 64157",4.0,2.0,...,147906.53,434100.0,NaN,NaN,10-807-00-20-006.00,Canceled,NaN,NaN,KCMO.10804_N_RANDOLPH_AVE_64157,Kansas City
4,Lafayette County,07-13-2026,11:30,2134 Pierson Ln,Bates City,64011,Keith W Teeny and Sherry Teeny,"Po Box 494 Oak Grove, MO 64075",3.0,NaN,...,296400.00,438500.0,NaN,NaN,24-6.0-23-0-000-013.000,Canceled,NaN,NaN,KCMO.2134_PIERSON_LN_64011,Kansas City


In [210]:
#STL

# Create APN column
stl["APN"] = (
    "STL_MO."
    + stl["Address"]
        .fillna("")
        .str.replace(" ", "_",regex = False)
        .str.upper()
    +"_"
    + stl["Zip"].astype(str)
)

#Create Market column
stl["Market"] = "St. Louis"


In [211]:
stl.head()

,County,Date,Time,Address,City,Zip,Owner Name,Mailing Address,Bedrooms,Bathrooms,...,Opening Bid,Assessed Value,Proj. Opening Bid,Original Loan Amount,APN Number,Status,Action,Comments,APN,Market


In [212]:
#Springfield

# Create APN column
spfd["APN"] = (
    "SPRINGFIELD_MO."
    + spfd["Address"]
        .fillna("")
        .str.replace(" ", "_",regex = False)
        .str.upper()
    +"_"
    + spfd["Zip"].astype(str)
)

#Create Market column
spfd["Market"] = "Springfield"

In [213]:
spfd.head()

,County,Date,Time,Address,City,Zip,Owner Name,Mailing Address,Bedrooms,Bathrooms,...,Opening Bid,Assessed Value,Proj. Opening Bid,Original Loan Amount,APN Number,Status,Action,Comments,APN,Market


#### Combine all data to one csv

In [214]:
davidimport = pd.concat([kc, stl, spfd], ignore_index=True)

davidimport.to_csv("export_data/davidimport.csv", index=False)

/var/folders/0v/2p48l9mx23dc0422gf24z_qr0000gn/T/ipykernel_92057/3750653891.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  davidimport = pd.concat([kc, stl, spfd], ignore_index=True)


In [215]:
#Find duplicate APNs

duplicate_apns = davidimport[
    davidimport.duplicated(subset="APN", keep=False)
]

# View them
duplicate_apns.sort_values("APN")

,County,Date,Time,Address,City,Zip,Owner Name,Mailing Address,Bedrooms,Bathrooms,...,Opening Bid,Assessed Value,Proj. Opening Bid,Original Loan Amount,APN Number,Status,Action,Comments,APN,Market
20,Jackson County,07-15-2026,14:00,10047 E Wilson Rd,Independence,64053,Blue Summit Enterprises LLC,"10047 E Wilson Rd, Independence, MO 64053",NaN,NaN,...,NaN,271600.0,41003.0,74500.0,14-730-18-04-01-0-00-000,Active,NaN,NaN,KCMO.10047_E_WILSON_RD_64053,Kansas City
21,Jackson County,07-15-2026,14:00,10047 E Wilson Rd,Independence,64053,Blue Summit Enterprises LLC,"10047 E Wilson Rd, Independence, MO 64053",NaN,NaN,...,NaN,271600.0,41003.0,74500.0,14-730-18-04-01-0-00-000,Active,NaN,NaN,KCMO.10047_E_WILSON_RD_64053,Kansas City


In [216]:
#Remove duplicate apns

davidimport.drop_duplicates(['APN'], keep = 'first', inplace=True)

#### Add Columns
County, Last Name

In [217]:
#Full address
#Labeled as 'Last Name' for Salesforce Lead object purposes

davidimport["Last Name"] = (
    davidimport["Address"].fillna("").astype(str)
    + ", "
    + davidimport["City"].fillna("").astype(str)
    + ", MO "
    + davidimport["Zip"].fillna("").astype(str)
)

davidimport["Last Name"] = davidimport["Last Name"]

In [218]:
davidimport.head()

,County,Date,Time,Address,City,Zip,Owner Name,Mailing Address,Bedrooms,Bathrooms,...,Assessed Value,Proj. Opening Bid,Original Loan Amount,APN Number,Status,Action,Comments,APN,Market,Last Name
0,Jackson County,07-13-2026,10:00,4070 SW Lido Dr,Lees Summit,64082,Jason T Medley and Heather Medley,"4070 SW Lido Dr, Lees Summit, MO 64082",3.0,2.0,...,415800.0,327328.0,330687.0,70-930-01-46-00-0-00-000,Canceled,NaN,NaN,KCMO.4070_SW_LIDO_DR_64082,Kansas City,"4070 SW Lido Dr, Lees Summit, MO 64082"
1,Cass County,07-13-2026,10:00,306 W Walnut St,Archie,64725,Cody S Thompson,"306 W Walnut St, Archie, MO 64725",2.0,NaN,...,165700.0,153317.0,157070.0,77500,Active,NaN,NaN,KCMO.306_W_WALNUT_ST_64725,Kansas City,"306 W Walnut St, Archie, MO 64725"
2,Clay County,07-13-2026,10:40,800 Southbrook Pkwy,Kearney,64060,Kurtis L. Springer,"800 Southbrook Pkwy, Kearney, MO 64060",3.0,2.0,...,315000.0,112584.0,137710.0,07-918-00-15-005.00,Canceled,NaN,NaN,KCMO.800_SOUTHBROOK_PKWY_64060,Kansas City,"800 Southbrook Pkwy, Kearney, MO 64060"
3,Clay County,07-13-2026,10:40,10804 N Randolph Ave,Kansas City,64157,Ronald G Smith and Janet M Smith,"10804 N Randolph Ave, Kansas City, MO 64157",4.0,2.0,...,434100.0,NaN,NaN,10-807-00-20-006.00,Canceled,NaN,NaN,KCMO.10804_N_RANDOLPH_AVE_64157,Kansas City,"10804 N Randolph Ave, Kansas City, MO 64157"
4,Lafayette County,07-13-2026,11:30,2134 Pierson Ln,Bates City,64011,Keith W Teeny and Sherry Teeny,"Po Box 494 Oak Grove, MO 64075",3.0,NaN,...,438500.0,NaN,NaN,24-6.0-23-0-000-013.000,Canceled,NaN,NaN,KCMO.2134_PIERSON_LN_64011,Kansas City,"2134 Pierson Ln, Bates City, MO 64011"


In [219]:
#County
#All of the values contain county at the end, so I want to remove that

davidimport["County"] = davidimport["County"].str.replace(" County", "", regex=False) 

In [220]:
#Change currency columns to numbers

for col in ["Opening Bid"]:
    davidimport.loc[:, col] = (
        davidimport[col]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .astype(float)
    )

In [221]:
davidimport.loc[
    davidimport["Status"] == "Canceled",
    ["Status", "Opening Bid"]
].head(20)

,Status,Opening Bid
0,Canceled,NaN
2,Canceled,NaN
3,Canceled,147906.53
4,Canceled,296400.00
12,Canceled,NaN
23,Canceled,34947.74
27,Canceled,NaN
29,Canceled,NaN
34,Canceled,NaN
49,Canceled,NaN


In [222]:
#Update Auction Status 

davidimport["Status"] = np.where(
    davidimport["Status"] == "Canceled",
    "Cancelled",
    ""
)

In [223]:
#Update Opening Bid

davidimport["Opening Bid"] = davidimport["Opening Bid"].astype("string")

davidimport.loc[
    davidimport["Status"].str.strip().str.upper() == "CANCELLED",
    "Opening Bid"
] = ""

In [224]:
davidimport.loc[
    davidimport["Status"] == "Cancelled",
    ["Status", "Opening Bid"]
].head(20)

,Status,Opening Bid
0,Cancelled,
2,Cancelled,
3,Cancelled,
4,Cancelled,
12,Cancelled,
23,Cancelled,
27,Cancelled,
29,Cancelled,
34,Cancelled,
49,Cancelled,


In [225]:
#Change column names
davidimport = davidimport.rename(columns={
    "Date": "Auction Date Updated",
    "Time": "Auction Sale Time",
    "Owner Name": "Property Owner Name",
    "Square Feet": "Bldg Sq Ft",
    "Trustee": "Auction Trustee",
    "File #": "NS Trustee Sale Number",
    "Opening Bid": "Auction Opening Bid",
    "Status": "Auction Status",
    "Bedrooms": "Beds",
    "Bathrooms": "Baths",
    "APN Number": "Tax ID"
})

In [226]:
davidimport.head()

,County,Auction Date Updated,Auction Sale Time,Address,City,Zip,Property Owner Name,Mailing Address,Beds,Baths,...,Assessed Value,Proj. Opening Bid,Original Loan Amount,Tax ID,Auction Status,Action,Comments,APN,Market,Last Name
0,Jackson,07-13-2026,10:00,4070 SW Lido Dr,Lees Summit,64082,Jason T Medley and Heather Medley,"4070 SW Lido Dr, Lees Summit, MO 64082",3.0,2.0,...,415800.0,327328.0,330687.0,70-930-01-46-00-0-00-000,Cancelled,NaN,NaN,KCMO.4070_SW_LIDO_DR_64082,Kansas City,"4070 SW Lido Dr, Lees Summit, MO 64082"
1,Cass,07-13-2026,10:00,306 W Walnut St,Archie,64725,Cody S Thompson,"306 W Walnut St, Archie, MO 64725",2.0,NaN,...,165700.0,153317.0,157070.0,77500,,NaN,NaN,KCMO.306_W_WALNUT_ST_64725,Kansas City,"306 W Walnut St, Archie, MO 64725"
2,Clay,07-13-2026,10:40,800 Southbrook Pkwy,Kearney,64060,Kurtis L. Springer,"800 Southbrook Pkwy, Kearney, MO 64060",3.0,2.0,...,315000.0,112584.0,137710.0,07-918-00-15-005.00,Cancelled,NaN,NaN,KCMO.800_SOUTHBROOK_PKWY_64060,Kansas City,"800 Southbrook Pkwy, Kearney, MO 64060"
3,Clay,07-13-2026,10:40,10804 N Randolph Ave,Kansas City,64157,Ronald G Smith and Janet M Smith,"10804 N Randolph Ave, Kansas City, MO 64157",4.0,2.0,...,434100.0,NaN,NaN,10-807-00-20-006.00,Cancelled,NaN,NaN,KCMO.10804_N_RANDOLPH_AVE_64157,Kansas City,"10804 N Randolph Ave, Kansas City, MO 64157"
4,Lafayette,07-13-2026,11:30,2134 Pierson Ln,Bates City,64011,Keith W Teeny and Sherry Teeny,"Po Box 494 Oak Grove, MO 64075",3.0,NaN,...,438500.0,NaN,NaN,24-6.0-23-0-000-013.000,Cancelled,NaN,NaN,KCMO.2134_PIERSON_LN_64011,Kansas City,"2134 Pierson Ln, Bates City, MO 64011"


#### Auction Trustee Lookups

In [227]:
contacts = pd.read_csv('sf_data/trusteecontacts.csv')

In [228]:
# Clean up column names if there are leading/trailing spaces
#contacts.columns = contacts.columns.str.strip()

In [229]:
#Normalize text for Contact Name and Associated Names

def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).upper()
    s = s.replace("\u00A0", " ")              # non-breaking spaces
    s = re.sub(r"[^\w\s]", " ", s)            # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()        # collapse spaces
    return s

contacts["contact_name_norm"] = contacts["Contact: Contact Name"].map(norm)
contacts["assoc_norm"] = contacts["Associated Names"].map(norm)  # can be blank

In [230]:
#contacts["contact_name_norm"] = contacts["Contact: Contact Name"]
#contacts["assoc_norm"] = contacts["Associated Names"]
#davidimport["trustee_norm"] = davidimport["Auction Trustee"]

In [231]:
#Normalize trustee names in the auction data

davidimport["trustee_norm"] = davidimport["Auction Trustee"].map(norm)

In [232]:
#Find exact matches

exact_map = dict(zip(contacts["contact_name_norm"], contacts["Contact: Contact Name"]))
davidimport["Auction Trustee (Lookup)"] = davidimport["trustee_norm"].map(exact_map)

In [233]:
#Find contains matches

def contains_in_contact_name(trustee_norm_value: str):
    if not trustee_norm_value:
        return None

    # find first contact where contact name contains trustee value
    hits = contacts.loc[
        contacts["contact_name_norm"].str.contains(re.escape(trustee_norm_value), na=False),
        "Contact: Contact Name"
    ]
    return hits.iloc[0] if len(hits) else None

mask = davidimport["Auction Trustee (Lookup)"].isna() & (davidimport["trustee_norm"] != "")
davidimport.loc[mask, "Auction Trustee (Lookup)"] = davidimport.loc[mask, "trustee_norm"].map(contains_in_contact_name)

In [234]:
#Find contains matches in associated names

def contains_in_associated(trustee_norm_value: str):
    if not trustee_norm_value:
        return None

    hits = contacts.loc[
        contacts["assoc_norm"].str.contains(re.escape(trustee_norm_value), na=False),
        "Contact: Contact Name"
    ]
    return hits.iloc[0] if len(hits) else None

mask = davidimport["Auction Trustee (Lookup)"].isna() & (davidimport["trustee_norm"] != "")
davidimport.loc[mask, "Auction Trustee (Lookup)"] = davidimport.loc[mask, "trustee_norm"].map(contains_in_associated)

In [235]:
#Collect and show unmatched rows

unmatched = (
    davidimport.loc[davidimport["Auction Trustee (Lookup)"].isna(),["Last Name", "Auction Trustee", "trustee_norm"]])

unmatched

,Last Name,Auction Trustee,trustee_norm
6,"5007 Red Barn Ln, St. Joseph, MO 64503","Bill Puett, Sheriff",BILL PUETT SHERIFF
20,"10047 E Wilson Rd, Independence, MO 64053","CHINNERY EVANS & NAIL, P.C",CHINNERY EVANS NAIL P C
26,"5812 NW Buttonwood Dr, Kansas City, MO 64152","singers corp, By:/s/ Michael P. Gaughan",SINGERS CORP BY S MICHAEL P GAUGHAN
33,"3294 Highway Oo, Odessa, MO 64076","singers corp, Michael P. Gaughan",SINGERS CORP MICHAEL P GAUGHAN
45,"319 Olive St, Kansas City, MO 64124","Austin B. Hayden CMP Agent Services, Inc",AUSTIN B HAYDEN CMP AGENT SERVICES INC
47,"23101 S State Route 291, Harrisonville, MO 64701",Michael J. Schmid,MICHAEL J SCHMID
48,"4300 E 75th Ter, Kansas City, MO 64132","MPOWB TRUSTEE SERVICES, LLC",MPOWB TRUSTEE SERVICES LLC
73,"2201 Vine St, Kansas City, MO 64108","Forsyth Property, Inc",FORSYTH PROPERTY INC
77,"925 Grand Blvd, Kansas City, MO 64106",CENTRE TRUSTEE CORP. By: Michele M. OMalley,CENTRE TRUSTEE CORP BY MICHELE M OMALLEY


In [236]:
#Confirm what should match but didnt

missing = (
    davidimport.loc[davidimport["Auction Trustee (Lookup)"].isna(), "trustee_norm"]
    .dropna().drop_duplicates()
)

# which of these are present as a normalized contact name?
present_mask = missing.isin(contacts["contact_name_norm"])
missing_not_in_contacts = missing[~present_mask]

missing_not_in_contacts.head(50)

6                           BILL PUETT SHERIFF
20                     CHINNERY EVANS NAIL P C
26         SINGERS CORP BY S MICHAEL P GAUGHAN
33              SINGERS CORP MICHAEL P GAUGHAN
45      AUSTIN B HAYDEN CMP AGENT SERVICES INC
47                            MICHAEL J SCHMID
48                  MPOWB TRUSTEE SERVICES LLC
73                        FORSYTH PROPERTY INC
77    CENTRE TRUSTEE CORP BY MICHELE M OMALLEY
Name: trustee_norm, dtype: object

In [237]:
davidimport.to_csv("export_data/davidimport.csv", index=False)